# 01 · Getiri ve risk

**Öğreneceklerin:** ham fiyatı analize hazır hale getirmek (bedelsiz düzeltmesi), log getiri, TL vs $
getiri, yıllık oynaklık, beta, en büyük düşüş (max drawdown), korelasyon.

**Nasıl çalışılır:** `TODO` hücrelerindeki fonksiyonları sen yazarsın; hemen altındaki `K.…` hücresi
kontrol eder ve ✅ basar. Takılırsan önce docstring'i tekrar oku, sonra bana ipucu sor.

Çalıştırma (repo kökünden): `uv run --group learn jupyter lab learn/`

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd()))  # learn/ klasöründen çalıştır
import kontrol as K
import papidata as pdx

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
plt.rcParams["figure.figsize"] = (11, 4)

In [ ]:
TICKERS = ["THYAO", "TUPRS", "CCOLA", "XU100", "TI1"]
px = pdx.prices(TICKERS)  # papi'nin SQLite'ından, ham kapanışlar
print(px.shape, px.index.min().date(), "→", px.index.max().date())
px.tail()

In [ ]:
ax = px[["THYAO", "TUPRS", "CCOLA"]].plot(logy=True, title="Ham fiyatlar (log ölçek)")
ax.set_ylabel("TL");

## 1. Bedelsiz sermaye artırımı: grafiğindeki uçurumlar

TUPRS'ta 2023'te, CCOLA'da 2024'te fiyat bir günde ~%85–90 düşüyor. Bu bir çöküş değil: **bedelsiz
sermaye artırımı**. Şirket %1000 bedelsiz verince her 1 payın 11 pay olur, fiyat da ~11'e bölünür;
yatırımcının serveti değişmez. papi fiyatları **ham** saklar, düzeltmek senin işin.

- Oran (`ratio`) = eski fiyat / yeni fiyat ≈ e^(−log getiri). %1000 bedelsiz → oran 11.
- Gerçek çöküşler de var: THYAO 2016-07-18'de %−12,5 (darbe girişimi sonrası). Fark: o gün **piyasa
  da** düştü (XU100 %−7). Bedelsizde piyasa normaldir.
- İpucu: hissenin getirisinden piyasanın getirisini çıkar; kalan "fazla getiri" çok büyükse kurumsal işlemdir.
- Temettü de fiyatı düşürür (TUPRS 2019-03-28 %−13, temettü günü). Onu şimdilik düzeltmiyoruz: bonus görev.

In [ ]:
def find_corporate_actions(p: pd.Series, market: pd.Series, threshold: float = 0.25) -> pd.DataFrame:
    """Bedelsiz/bölünme günlerini bul.

    p: bir hissenin ham kapanışları, market: XU100 kapanışları (aynı tarih ekseni).
    Döndür: index = kurumsal işlem günü, sütunlar:
      - log_ret: o günkü hissenin log getirisi
      - ratio:   eski fiyat / yeni fiyat (= e^(−log_ret))
    Hiç yoksa boş DataFrame (aynı sütunlarla).
    """
    # TODO
    raise NotImplementedError

In [ ]:
K.find_corporate_actions(find_corporate_actions, px)

In [ ]:
def adjust_prices(p: pd.Series, actions: pd.DataFrame) -> pd.Series:
    """Geriye doğru düzelt: her kurumsal işlem gününden ÖNCEKİ fiyatları o günün `ratio`'suna böl.

    Son fiyat aynı kalır (bugünkü fiyatla karşılaştırılabilir seri). Birden fazla işlem varsa hepsini uygula.
    """
    # TODO
    raise NotImplementedError

In [ ]:
K.adjust_prices(adjust_prices, find_corporate_actions, px)

In [ ]:
px_adj = px.copy()
for t in ["THYAO", "TUPRS", "CCOLA"]:
    s = px[t].dropna()
    acts = find_corporate_actions(s, px["XU100"])
    px_adj[t] = adjust_prices(s, acts)
    print(t, "kurumsal işlemler:", [(d.date(), round(r, 2)) for d, r in acts["ratio"].items()])
px_adj[["THYAO", "TUPRS", "CCOLA"]].plot(logy=True, title="Düzeltilmiş fiyatlar (log ölçek)");

## 2. Log getiri

Basit getiri r = P_t / P_{t−1} − 1, log getiri ℓ = ln(P_t / P_{t−1}). Log getirinin güzelliği:
**toplanabilir**. 5 günün log getirisi = 5 günlük log getirilerin toplamı. Basit getiri çarpılarak birleşir.
Küçük hareketlerde ikisi neredeyse aynıdır; büyük hareketlerde ayrışır (+%50 sonra −%50 = −%25!).

In [ ]:
def log_returns(df: pd.DataFrame) -> pd.DataFrame:
    """Her sütun için günlük log getiri. Aynı şekil; ilk satır NaN."""
    # TODO
    raise NotImplementedError

In [ ]:
K.log_returns(log_returns, px_adj)
r = log_returns(px_adj)

## 3. TL mi, dolar mı?

TL'de 10 yılda %1000 getiri etkileyici görünür; ama aynı dönemde USD/TRY ~2,3'ten ~50'ye çıktı.
Yabancı yatırımcı ve çoğu analist BIST'i **$ bazında** da değerlendirir. Reel (enflasyondan arındırılmış)
getiri de bir seçenek; burada $ kullanıyoruz çünkü günlük veri var.

In [ ]:
def to_usd(df: pd.DataFrame, usdtry: pd.Series) -> pd.DataFrame:
    """TL fiyatları $'a çevir: fiyat / o günkü USD/TRY.

    Kur serisinde fiyat günü yoksa (tatil, farklı takvim) ondan önceki son kuru kullan (forward-fill).
    Index = df.index aynen korunmalı.
    """
    # TODO
    raise NotImplementedError

In [ ]:
usdtry = pdx.usdtry()
K.to_usd(to_usd, px, usdtry)
px_usd = to_usd(px_adj, usdtry)
r_usd = log_returns(px_usd)

## 4. Risk: oynaklık, beta, en büyük düşüş

- **Yıllık oynaklık** = günlük log getirilerin standart sapması × √252 (yılda ~252 işlem günü).
  Neden √? Bağımsız günlerin varyansı toplanır; std, varyansın kökü.
- **Beta** = Cov(hisse, piyasa) / Var(piyasa). Piyasa %1 hareket edince hisse ortalama ne kadar hareket ediyor?
- **Max drawdown** = bir zirveden sonraki en derin düşüş. Oynaklıktan daha "hissedilen" risk ölçüsü.

In [ ]:
def annual_vol(r: pd.Series, periods: int = 252) -> float:
    """Yıllıklandırılmış oynaklık (NaN'ları at)."""
    # TODO
    raise NotImplementedError


def beta(r_asset: pd.Series, r_market: pd.Series) -> float:
    """Cov(asset, market) / Var(market); sadece ikisinin de dolu olduğu günler."""
    # TODO
    raise NotImplementedError


def max_drawdown(p: pd.Series) -> tuple[float, pd.Timestamp, pd.Timestamp]:
    """(en büyük düşüş oranı (negatif, ör. −0.5), zirve tarihi, dip tarihi).

    İpucu: o güne kadarki en yüksek fiyat = p.cummax().
    """
    # TODO
    raise NotImplementedError

In [ ]:
K.annual_vol(annual_vol, r)
K.beta(beta, r)
K.max_drawdown(max_drawdown)

In [ ]:
def cagr(p: pd.Series) -> float:
    p = p.dropna()
    years = (p.index[-1] - p.index[0]).days / 365.25
    return (p.iloc[-1] / p.iloc[0]) ** (1 / years) - 1

since = "2021-07-01"  # TI1 dahil hepsinin verisi olan dönem
rows = {}
for t in TICKERS:
    mdd, peak, trough = max_drawdown(px_usd[t][since:].dropna())
    rows[t] = {
        "CAGR TL": cagr(px_adj[t][since:]),
        "CAGR $": cagr(px_usd[t][since:]),
        "oynaklık $": annual_vol(r_usd[t][since:]),
        "beta (TL, XU100)": beta(r[t][since:], r["XU100"][since:]),
        "max düşüş $": mdd,
        "dip": trough.date(),
    }
summary = pd.DataFrame(rows).T
summary

In [ ]:
corr = r_usd[since:].corr()
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(corr, vmin=-1, vmax=1, cmap="coolwarm")
ax.set_xticks(range(len(corr)), corr.columns, rotation=45)
ax.set_yticks(range(len(corr)), corr.columns)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
fig.colorbar(im); ax.set_title("Günlük $ getiri korelasyonu");

## 5. Yorumla (cevapları bu hücreye yaz)

1. TL ve $ bazında CAGR arasındaki fark neyi gösteriyor? Hangi hisse $ bazında gerçekten kazandırdı?
2. TI1'in oynaklığı neden bu kadar düşük, $ bazında neden kaybettiriyor olabilir?
3. Betası en yüksek hisse hangisi? Bu, 30 Ekim gibi piyasanın %3,6 düştüğü günlerde ne demek?
4. Korelasyon matrisinde en düşük korelasyon hangi çiftte? Çeşitlendirme açısından ne anlama gelir?
5. **Bonus:** Temettüleri de düzelterek "toplam getiri" serisi kur. (İpucu: KAP'ta temettü bildirimleri;
   temettü gününden önceki tüm fiyatları (1 − temettü / önceki kapanış) ile çarp.)

*Cevaplar:*

In [ ]:
pdx.DATA_DIR.mkdir(exist_ok=True)
px_adj.to_csv(pdx.DATA_DIR / "prices_adj.csv")  # 02 ve 04 bunu kullanır
print("kaydedildi:", pdx.DATA_DIR / "prices_adj.csv")

## papi'ye taşıma (sonra)

Bedelsiz düzeltmesi papi'nin M3 hedefinde var ("corporate-action adjustment"). Notebook'ta çalışan
`find_corporate_actions` + `adjust_prices`'ı ileride `src/papi/sim/` altına test'leriyle taşıyacaksın;
o zaman tahmin yerine KAP'taki gerçek bedelsiz oranlarını (`corporate_action` tablosu) kullanmak daha doğru.